# Filtrado VCF — Genes Candidatos MVA

**Track 2 — Reposicionamiento de Fármacos · MVA Hackathon 2026**

Este notebook filtra las 5,012,204 variantes del WGS para quedarse solo con las que caen en genes implicados en MVA, y realiza una priorización inicial por zigosidad.

**Genes candidatos:**

| Gen | Función | Relevancia MVA |
|-----|---------|----------------|
| **BUB1B** | Codifica BUBR1 — componente central del SAC (kinetocoro) | ~50% de casos MVA; principal candidato |
| **CEP57** | Proteína centrosómica que estabiliza microtúbulos | MVA tipo 2 |
| **TRIP13** | ATPasa que disuelve el Mitotic Checkpoint Complex (MCC) | MVA recién descrito |
| BUB1 | Kinasa del SAC, recluta BUB3 y MAD1/MAD2 | SAC accesorio |
| MAD1L1 | Componente del complejo de espera del SAC | Gen muy grande (418 kb) |
| MAD2L1 | Inhibidor de APC/C en el SAC | Gen pequeño, variantes más interpretables |

**Herencia esperada:** autosómica recesiva → buscaremos **variantes homocigotas** o **heterocigóticas compuestas** en los genes primarios.

## 0. Setup

In [ ]:
import gzip
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter

DATA_DIR    = Path("../data/raw")
RESULTS_DIR = Path("../results")
VCF         = DATA_DIR / "WGS_EX2312012_HGWCNDSX7.vcf.gz"
RESULTS_DIR.mkdir(exist_ok=True)

print("Entorno listo.")

## 1. Coordenadas GRCh38 de los genes candidatos

Consultamos la API REST de Ensembl. Si falla (sin conexión), se usan coordenadas de respaldo verificadas.

> **Nota:** el VCF usa cromosomas sin prefijo `chr` (ej. `15`, no `chr15`) — la región `seq_region_name` de Ensembl coincide directamente.

In [ ]:
# Coordenadas de respaldo (verificadas contra Ensembl GRCh38, febrero 2025)
FALLBACK_COORDS = {
    "BUB1B":  ("15", 40_160_984,  40_221_137),
    "CEP57":  ("11", 95_789_965,  95_837_070),
    "TRIP13": ("5",    892_849,      919_357),
    "BUB1":   ("2",  110_143_783, 110_244_769),
    "MAD1L1": ("7",   1_815_787,   2_233_243),
    "MAD2L1": ("4",  120_055_623, 120_067_192),
}
PADDING = 5_000  # bp extra a cada lado para capturar regiones regulatorias próximas

def fetch_ensembl(gene):
    url = f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene}?content-type=application/json"
    try:
        d = requests.get(url, timeout=10).json()
        return str(d["seq_region_name"]), int(d["start"]), int(d["end"])
    except Exception:
        return None

gene_coords = {}
for gene, fb in FALLBACK_COORDS.items():
    result = fetch_ensembl(gene)
    if result:
        gene_coords[gene] = result
        print(f"{gene:8s}  {result[0]}:{result[1]:>12,}-{result[2]:>12,}  (Ensembl)")
    else:
        gene_coords[gene] = fb
        print(f"{gene:8s}  {fb[0]}:{fb[1]:>12,}-{fb[2]:>12,}  (fallback)")

with open(RESULTS_DIR / "mva_gene_coords_grch38.json", "w") as f:
    json.dump(gene_coords, f, indent=2)

## 2. Filtrado del VCF

Criterios aplicados:
- **FILTER = PASS** (pasa los filtros de calidad GATK)
- **Posición dentro del intervalo de un gen candidato** ± 5 kb

In [ ]:
# Construir índice cromosoma → lista de (start, end, gen)
interval_idx = {}
for gene, (chrom, start, end) in gene_coords.items():
    interval_idx.setdefault(chrom, []).append((start - PADDING, end + PADDING, gene))

def gene_at(chrom, pos):
    for s, e, g in interval_idx.get(chrom, []):
        if s <= pos <= e:
            return g
    return None

def classify_zygosity(gt):
    alleles = gt.replace("|", "/").split("/")
    if "." in alleles:
        return "MISSING"
    if set(alleles) == {"0"}:
        return "HOM_REF"
    if alleles[0] == alleles[1] and alleles[0] != "0":
        return "HOM_ALT"
    if "0" in alleles:
        return "HET"
    return "OTHER"

hits = []
print(f"Escaneando {VCF.name} ...")

with gzip.open(VCF, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        p = line.rstrip().split("\t")
        chrom, pos_s, vid, ref, alt, qual, filt, info, fmt = p[:9]
        sample = p[9] if len(p) > 9 else ""
        if filt != "PASS":
            continue
        pos = int(pos_s)
        gene = gene_at(chrom, pos)
        if not gene:
            continue
        gt_dict = dict(zip(fmt.split(":"), sample.split(":")))
        gt  = gt_dict.get("GT", ".")
        dp  = int(gt_dict.get("DP", 0) or 0)
        gq  = int(gt_dict.get("GQ", 0) or 0)
        ad  = gt_dict.get("AD", ".")
        var_type = "SNV" if len(ref) == 1 and len(alt) == 1 else (
                   "DEL" if len(ref) > len(alt) else "INS")
        hits.append({
            "gene": gene, "chrom": chrom, "pos": pos,
            "id": vid, "ref": ref, "alt": alt, "qual": float(qual),
            "type": var_type, "zygosity": classify_zygosity(gt),
            "GT": gt, "DP": dp, "GQ": gq, "AD": ad,
        })

df = pd.DataFrame(hits)
df.to_csv(RESULTS_DIR / "variants_mva_genes.tsv", sep="\t", index=False)
print(f"Total variantes PASS en genes candidatos: {len(df):,}")
df.head()

## 3. Resumen estadístico

### Resultado del filtrado

De 5,012,204 variantes totales en el WGS, **1,075 pasan los filtros de calidad y caen en los 6 genes candidatos**:

| Gen | Variantes PASS | Homocigotas ALT |
|-----|---------------|------------------|
| MAD1L1 | 901 | 204 |
| BUB1 | 112 | 8 |
| MAD2L1 | 36 | 7 |
| **BUB1B** | **16** | **7** |
| **CEP57** | **7** | **3** |
| TRIP13 | 3 | 0 |

In [ ]:
print("=== Variantes por gen ===")
print(df.groupby("gene")[["pos"]].count().rename(columns={"pos": "total"}).sort_values("total", ascending=False))

print("\n=== Por gen y zigosidad ===")
print(pd.crosstab(df["gene"], df["zygosity"]))

print("\n=== Por tipo de variante ===")
print(df.groupby(["gene", "type"])["pos"].count().unstack(fill_value=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel izquierdo: variantes por gen y zigosidad (stacked bar)
ct = pd.crosstab(df["gene"], df["zygosity"])
gene_order = ct.sum(axis=1).sort_values(ascending=False).index
ct.loc[gene_order].plot(kind="bar", stacked=True, ax=axes[0],
    color={"HET": "#4C72B0", "HOM_ALT": "#C44E52", "OTHER": "#8C8C8C", "MISSING": "#BBBBBB"},
    width=0.6)
axes[0].set_title("Variantes PASS por gen y zigosidad")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
axes[0].set_ylabel("Número de variantes")

# Panel derecho: solo genes primarios BUB1B, CEP57, TRIP13 — zigosidad
primary = df[df["gene"].isin(["BUB1B", "CEP57", "TRIP13"])]
ct2 = pd.crosstab(primary["gene"], primary["zygosity"])
ct2.plot(kind="bar", stacked=True, ax=axes[1],
    color={"HET": "#4C72B0", "HOM_ALT": "#C44E52", "OTHER": "#8C8C8C"},
    width=0.5)
axes[1].set_title("Genes primarios MVA — detalle")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "variants_by_gene_zygosity.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Variantes homocigotas en genes primarios

La herencia autosómica recesiva de MVA implica que la variante causante debe encontrarse en **homocigosis** (ambas copias del gen afectadas por la misma variante) o **heterocigosis compuesta** (dos variantes diferentes, una en cada copia). Comenzamos con las homocigotas.

> ⚠️ **Importante:** Muchas variantes homocigotas son polimorfismos comunes (alta frecuencia en gnomAD). El siguiente paso es cruzarlas con bases de datos de frecuencia poblacional para quedarnos solo con las **raras** (AF < 1% en gnomAD).

In [ ]:
# Homocigotas en los 3 genes primarios
primary_genes = ["BUB1B", "CEP57", "TRIP13"]
df_hom_primary = df[
    (df["gene"].isin(primary_genes)) &
    (df["zygosity"] == "HOM_ALT")
].sort_values(["gene", "pos"]).reset_index(drop=True)

print(f"Variantes homocigotas en BUB1B / CEP57 / TRIP13: {len(df_hom_primary)}")
display(df_hom_primary[["gene", "chrom", "pos", "ref", "alt", "type", "qual", "DP", "GQ", "id"]])

### Hallazgos destacados — variantes homocigotas

**BUB1B** (7 variantes homocigotas):

| Posición (GRCh38) | Cambio | Tipo | DP | GQ | ID |
|------------------|--------|------|----|----|----|
| 15:40,158,208 | C>G | SNV | 40 | 99 | — |
| 15:40,186,692 | G>A | SNV | 41 | 99 | — |
| 15:40,187,404 | A>G | SNV | 47 | 99 | — |
| 15:40,193,435 | T>C | SNV | 42 | 99 | — |
| 15:40,198,239 | G>GT | INS | 36 | 99 | — |
| 15:40,216,568 | TA>T | DEL | 26 | 78 | — |
| 15:40,216,570 | TA>T | DEL | 26 | 78 | — |

**CEP57** (3 variantes homocigotas):

| Posición | Cambio | Tipo | DP | GQ |
|----------|--------|------|----|----|  
| 11:95,787,319 | C>T | SNV | 31 | 93 |
| 11:95,793,416 | T>TTA | INS | 35 | 99 |
| 11:95,795,499 | C>CT | INS | 50 | 99 |

**TRIP13**: 3 variantes totales, ninguna homocigótica (las 3 son heterocigotas).

---

### Observación sobre MAD1L1

MAD1L1 tiene 204 variantes homocigotas distribuidas a lo largo de ~418 kb. Este patrón es característico de un **bloque de homocigosidad (ROH — Run of Homozygosity)**: ambas copias del cromosoma 7 son idénticas en esa región, posiblemente por consanguinidad parental. Las abortos espontáneos recurrentes en los padres son consistentes con consanguinidad. Este hallazgo debe documentarse aunque no implique que MAD1L1 sea la causa.

In [ ]:
# Visualizar distribución posicional de las homocigotas en MAD1L1 (evidencia de ROH)
df_mad1 = df[(df["gene"] == "MAD1L1") & (df["zygosity"] == "HOM_ALT")]

fig, ax = plt.subplots(figsize=(12, 3))
ax.scatter(df_mad1["pos"] / 1e6, [1]*len(df_mad1), s=8, alpha=0.5, color="#C44E52")
ax.set_xlabel("Posición en chr7 (Mb)")
ax.set_title("MAD1L1 — distribución de variantes homocigotas\n(patrón consistente con bloque ROH — posible consanguinidad parental)")
ax.set_yticks([])
ax.set_xlim(1.80, 2.24)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "mad1l1_roh_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Heterocigotas en BUB1B y CEP57 — búsqueda de compuestos

Para heterocigosis compuesta necesitamos ≥ 2 variantes heterocigotas en el mismo gen (una en cada alelo).

In [ ]:
for gene in ["BUB1B", "CEP57", "TRIP13"]:
    het = df[(df["gene"] == gene) & (df["zygosity"] == "HET")]
    print(f"\n{gene} — variantes heterocigotas ({len(het)}):")
    if len(het) > 0:
        display(het[["chrom", "pos", "ref", "alt", "type", "qual", "DP", "GQ", "id"]].reset_index(drop=True))
    else:
        print("  Ninguna.")

## 6. Próximo paso: priorización por frecuencia poblacional

El filtrado por coordenadas genómicas nos da **1,075 variantes candidatas**, la mayoría probablemente polimorfismos comunes. Para identificar la variante causante necesitamos:

```
Variantes PASS en genes MVA (1,075)
        │
        ▼  Cruzar con gnomAD v4 (API)
Filtrar AF < 0.01 en todas las poblaciones
        │
        ▼  Aplicar criterios funcionales
LOF (frameshift / nonsense / splice ±2)
ó missense con CADD ≥ 20
        │
        ▼
Candidatos de alta prioridad
        │
        ▼
Verificar en ClinVar + literatura MVA
```

**Notebook siguiente:** `03_priorizacion_gnomad.ipynb`

Consultará la API GraphQL de gnomAD v4 para cada variante y filtrará por frecuencia poblacional rara.